# 01 — Validate SAE (thin wrapper)
All logic lives in `interp_core/` + `audit/`. This notebook only calls it.

In [ ]:
import sys
sys.path.insert(0, '..')
from audit.extract import folder_images, gate_activations, set_seed
from interp_core.loaders import load_model, resolve_hook_name
from interp_core.sae import load_sae, validate_sae
from audit.report import write_gate_card

SAE_REPO = 'Prisma-Multimodal/sae-top_k-64-cls_only-layer_9-hook_resid_post'
SAE_REVISION = None  # pin to a commit sha for a publishable run
DATA = '/kaggle/input/imagenet-val-2k'
assert __import__('os').path.isdir(DATA), f'mount {DATA} first'
set_seed(1337)
bundle = load_model('open_clip:ViT-B-32')
sae = load_sae(SAE_REPO, revision=SAE_REVISION)
hook = resolve_hook_name(bundle.spec, sae.hook_layer, sae.hook_component)
images = folder_images(DATA, n=500)
acts = gate_activations(images, bundle, sae, bundle.preprocess, hook)
gate = validate_sae(acts, sae)
print(gate)
write_gate_card(gate, '/kaggle/working/out', hook)